# 20_06_state_dict_の保存と読み込み

コード：`ch20/blocks/20_06_state_dict_の保存と読み込み.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 付録 A-6：保存と読み込み。重み（state_dict）だけを保存し、同じ形のモデルに読み込む
import torch
from torch import nn

def make():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))

torch.manual_seed(0)
model = make()
torch.save(model.state_dict(), "weights.pt")              # 推奨：重みの辞書だけを保存する
print("保存した中身", list(torch.load("weights.pt").keys())[:2], "…")

clone = make()                                            # 形だけ同じ、新しいモデル
clone.load_state_dict(torch.load("weights.pt"))           # 重みを読み込む
x = torch.rand(2, 1, 28, 28)
print("読み込んだモデルと元のモデルの出力の差", (model(x) - clone(x)).abs().max().item())

# モデルの丸ごとの保存は、読み込みで止まる（既定では重みしか読み込まない）
torch.save(model, "whole.pt")
try:
    torch.load("whole.pt")
    print("丸ごとの読み込み：通った")
except Exception as e:
    print("丸ごとの読み込み：", type(e).__name__)
print("torch", torch.__version__)
